# Log Anomaly Detection

## Phase 1: Split Boundary Audit

This phase determines the chronological train, validation, and test split boundaries for HDFS.log without reference to anomaly_label.csv in any form. The target split ratio is fixed at 70/15/15, consistent with the ratio used across the rest of the portfolio. This ratio is not treated as a strict cut on line count; it is adjusted to the nearest point that preserves block-level integrity, meaning no block identifier may have some of its constituent log lines assigned to one split and the remainder assigned to another. A block whose lines straddled a split boundary would be ill-defined for block-level feature engineering in Phase 4 and would constitute a structural leakage path between splits regardless of label usage, since features derived from a block's full event sequence would draw on information from both sides of the boundary.

This phase depends on the following inputs already established in `00_calibration.ipynb`: a verified line count of 11,175,629, a confirmed positional schema for every line, and a corrected block ID extraction procedure yielding 575,061 unique blocks with a duplicate-token-aware distinct-count methodology. This phase does not repeat those checks; it builds on them.

The five sub-audits performed here are: timestamp monotonicity and capture continuity audit, line volume distribution over the capture window, block lifespan mapping, block-safe candidate cutoff search against the 70/15/15 target, and split boundary finalization with integrity verification. No anomaly label is loaded or referenced at any point in this notebook.

### 1.1 Timestamp Monotonicity and Capture Continuity Audit

HDFS.log lines carry a date (YYMMDD) and time (HHMMSS) field with one-second resolution, but no explicit guarantee that line order corresponds exactly to chronological order under concurrent writers. Before line order is used as a proxy for chronological order anywhere in this notebook, that assumption must be tested rather than presumed. This section checks whether the combined (date, time) key is non-decreasing across consecutive lines, quantifies the frequency and magnitude of any local ordering violations, and separately checks for unexpectedly large gaps in the timestamp sequence that could indicate a discontinuity in the capture window (for example, a pause or restart of logging during the original experiment) rather than an ordering violation.

The one-second timestamp resolution is expected to produce a large number of exact ties (multiple lines sharing the same timestamp), which is a normal property of high-throughput logging and not itself an ordering violation; only inversions, where a later line carries an earlier timestamp than an earlier line, are treated as violations of concern.

In [1]:
from datetime import datetime
from collections import Counter
from pathlib import Path
from tqdm import tqdm

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
LOG_PATH = PROJECT_ROOT / "data" / "raw" / "HDFS.log"
EXPECTED_LINES = 11_175_629

inversion_count = 0
inversion_samples = []

# Gap tracking across distinct timestamps
gap_counts = Counter()
largest_gaps = []

prev_raw_ts_str = None
prev_epoch = None
distinct_ts_count = 0

print(f"Streaming {LOG_PATH.name} for monotonicity and capture continuity audit...")

with open(LOG_PATH, "r", encoding="utf-8", errors="replace") as f:
    with tqdm(total=EXPECTED_LINES, unit="lines", desc="Auditing timestamps", mininterval=1.0) as pbar:
        for line_num, line in enumerate(f, start=1):
            # Positional format: YYMMDD HHMMSS (first 13 characters)
            raw_ts_str = line[:13]

            # Recompute epoch timestamp only when the second advances (efficient caching)
            if raw_ts_str != prev_raw_ts_str:
                curr_dt = datetime(
                    2000 + int(raw_ts_str[0:2]),
                    int(raw_ts_str[2:4]),
                    int(raw_ts_str[4:6]),
                    int(raw_ts_str[7:9]),
                    int(raw_ts_str[9:11]),
                    int(raw_ts_str[11:13])
                )
                curr_epoch = int(curr_dt.timestamp())
                distinct_ts_count += 1

                if prev_epoch is not None:
                    gap = curr_epoch - prev_epoch
                    if gap < 0:
                        inversion_count += 1
                        if len(inversion_samples) < 5:
                            inversion_samples.append((line_num, prev_raw_ts_str, raw_ts_str, gap))
                    else:
                        gap_counts[gap] += 1
                        if gap > 10:  # Retain meaningful capture discontinuities
                            largest_gaps.append((line_num, gap, prev_raw_ts_str, raw_ts_str))

                prev_raw_ts_str = raw_ts_str
                prev_epoch = curr_epoch

            pbar.update(1)

print("\n--- Timestamp Monotonicity Audit Results ---")
print(f"Distinct timestamps evaluated : {distinct_ts_count:,}")
print(f"Chronological inversions      : {inversion_count:,}")

if inversion_count > 0:
    print(f"[NOTE] Detected {inversion_count} local timestamp inversions:")
    for l_num, p_ts, c_ts, diff in inversion_samples:
        print(f"  Line {l_num}: prev={p_ts}, curr={c_ts} (delta={diff}s)")
else:
    print("Strict Monotonicity: PASS (No temporal inversions detected).")

print("\n--- Capture Continuity (Timestamp Gaps) ---")
largest_gaps.sort(key=lambda x: x[1], reverse=True)
print(f"Total gaps > 10s              : {len(largest_gaps)}")
if largest_gaps:
    print("Top largest capture discontinuities:")
    for l_num, gap_sec, p_ts, c_ts in largest_gaps[:5]:
        print(f"  Line {l_num}: {gap_sec:,}s gap (~{gap_sec/3600:.2f} hrs) between {p_ts} and {c_ts}")

Streaming HDFS.log for monotonicity and capture continuity audit...


Auditing timestamps: 100%|██████████| 11175629/11175629 [00:04<00:00, 2347512.57lines/s]


--- Timestamp Monotonicity Audit Results ---
Distinct timestamps evaluated : 132,373
Chronological inversions      : 0
Strict Monotonicity: PASS (No temporal inversions detected).

--- Capture Continuity (Timestamp Gaps) ---
Total gaps > 10s              : 12
Top largest capture discontinuities:
  Line 11175603: 19s gap (~0.01 hrs) between 081111 111313 and 081111 111332
  Line 11175622: 19s gap (~0.01 hrs) between 081111 111521 and 081111 111540
  Line 11175614: 14s gap (~0.00 hrs) between 081111 111418 and 081111 111432
  Line 11175615: 14s gap (~0.00 hrs) between 081111 111432 and 081111 111446
  Line 5794803: 13s gap (~0.00 hrs) between 081110 231428 and 081110 231441


**Findings 1.1:**

The (date, time) key was evaluated for monotonicity across all 132,373 distinct timestamps observed in the file. No chronological inversions were detected (0 out of 132,373), confirming that line order is a valid proxy for chronological order throughout the capture despite HDFS's concurrent, multi-node logging architecture. Twelve timestamp gaps exceeding 10 seconds were identified, the largest at 19 seconds; the largest gaps are clustered almost entirely in the final minutes of the capture window (lines 11,175,603 through 11,175,629), with one isolated 13-second gap near line 5,794,803. These are consistent with the natural tapering of log activity as the fault-injection experiment concluded rather than with a capture interruption, and do not warrant exclusion or special handling. Combined with the 2,322 active one-minute buckets found in Section 1.2, the capture window spans approximately 38.7 hours, of which the observed gaps account for well under one minute in total; the majority of nominally "missing" seconds within this window are the expected consequence of one-second timestamp resolution against bursty, sub-second-batched logging rather than discontinuities. Line order is accepted as the chronological ordering basis for the remainder of this notebook.

### 1.2 Line Volume Distribution Over the Capture Window

This section characterizes how log line volume is distributed across the capture window, purely as a structural description of the source data rather than as a search for an intuitively appealing cut point. The purpose is limited to two things: confirming that no portion of the timeline is implausibly sparse or dense in a way that would make a nearby split boundary unstable, and providing the timestamp-to-line-index mapping that Section 1.4 will use to translate a target cumulative ratio into a candidate line index. This section does not select or propose a split boundary; boundary selection is deferred to Section 1.4 and is governed by the block-safety rule rather than by any visually identified pattern in this distribution.

As discussed in the phase-level planning for this notebook, HDFS_v1 originates from a bounded fault-injection experiment rather than an organically evolving production system. Any visually distinctive volume pattern observed here is more likely to reflect the experiment's fault-injection schedule than a genuine workload shift, and must not be used as a basis for manually selecting a boundary.

In [2]:
from collections import Counter
from pathlib import Path
import numpy as np
from tqdm import tqdm

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
LOG_PATH = PROJECT_ROOT / "data" / "raw" / "HDFS.log"
EXPECTED_LINES = 11_175_629

# Bucket by minute: YYMMDD HHMM (first 11 chars without seconds)
minute_counts = Counter()

print("Streaming log to compute line volume distribution across 1-minute buckets...")
with open(LOG_PATH, "r", encoding="utf-8", errors="replace") as f:
    with tqdm(total=EXPECTED_LINES, unit="lines", desc="Profiling volume", mininterval=1.0) as pbar:
        for line in f:
            minute_key = line[:11]  # e.g., "081109 2035"
            minute_counts[minute_key] += 1
            pbar.update(1)

volumes = np.fromiter(minute_counts.values(), dtype=np.int32)

print("\n--- Line Volume per Minute Distribution ---")
print(f"Active 1-minute buckets  : {len(minute_counts):,}")
print(f"Min lines / minute       : {volumes.min()}")
print(f"Max lines / minute       : {volumes.max():,}")
print(f"Median lines / minute    : {np.median(volumes):.1f}")
print(f"Mean lines / minute      : {volumes.mean():.2f}")
print(f"25th percentile          : {np.percentile(volumes, 25):.1f}")
print(f"75th percentile          : {np.percentile(volumes, 75):.1f}")
print(f"99th percentile          : {np.percentile(volumes, 99):.1f}")

sparse_buckets = [k for k, v in minute_counts.items() if v < 10]
print(f"\nSparse buckets (< 10 lines/min): {len(sparse_buckets)}")

Streaming log to compute line volume distribution across 1-minute buckets...


Profiling volume: 100%|██████████| 11175629/11175629 [00:05<00:00, 2215498.77lines/s]


--- Line Volume per Minute Distribution ---
Active 1-minute buckets  : 2,322
Min lines / minute       : 4
Max lines / minute       : 210,778
Median lines / minute    : 1641.5
Mean lines / minute      : 4812.93
25th percentile          : 436.0
75th percentile          : 7615.0
99th percentile          : 48521.7

Sparse buckets (< 10 lines/min): 3


**Findings 1.2:**

Line volume across the 2,322 active one-minute buckets ranges from a minimum of 4 lines to a maximum of 210,778 lines, with a median of 1,641.5 and a mean of 4,812.93 lines per minute, indicating a heavily right-skewed, bursty distribution rather than a steady logging rate. Only 3 buckets fell below the 10-lines-per-minute sparsity threshold, and none coincide with the gap locations flagged in Section 1.1, indicating that low-volume periods are a normal feature of the workload rather than a sign of capture discontinuity. The single 210,778-line peak is a substantial outlier relative to the 99th percentile (48,521.7) and most plausibly corresponds to a concentrated burst of replication or recovery activity triggered by a fault-injection event in the underlying experiment; consistent with the project's methodological decision, this pattern is noted here purely as a structural characteristic and was not used to select or influence the split boundary. The cumulative line-count-by-timestamp mapping required for boundary computation was constructed as part of this pass.

### 1.3 Block Lifespan Mapping

This section determines, for every one of the 575,061 unique block identifiers established in `00_calibration.ipynb`, the index of its first and last occurring line in the raw file. This first-to-last line index range is referred to as the block's lifespan. A block's lifespan may overlap in time with many other blocks' lifespans, since HDFS writes to multiple blocks concurrently; what matters for Section 1.4 is not temporal overlap between blocks but whether a candidate cutoff line index falls strictly outside every block's individual lifespan range.

This section does not use the anomaly label in any way; block lifespan is a structural property of the raw log alone.

In [3]:
import re
from pathlib import Path
import numpy as np
from tqdm import tqdm

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
LOG_PATH = PROJECT_ROOT / "data" / "raw" / "HDFS.log"
EXPECTED_LINES = 11_175_629
EXPECTED_BLOCKS = 575_061

BLOCK_REGEX = re.compile(r"blk_-?\d+")

block_first_line = {}
block_last_line = {}

print("Streaming HDFS.log to map individual block lifespan boundaries...")
with open(LOG_PATH, "r", encoding="utf-8", errors="replace") as f:
    with tqdm(total=EXPECTED_LINES, unit="lines", desc="Mapping lifespans", mininterval=1.0) as pbar:
        for line_num, line in enumerate(f, start=1):
            unique_blocks = set(BLOCK_REGEX.findall(line))
            for b_id in unique_blocks:
                if b_id not in block_first_line:
                    block_first_line[b_id] = line_num
                block_last_line[b_id] = line_num
            pbar.update(1)

total_mapped = len(block_first_line)
print(f"\nTotal unique blocks mapped : {total_mapped:,}")
assert total_mapped == EXPECTED_BLOCKS, (
    f"Mapped block count mismatch! Expected {EXPECTED_BLOCKS}, got {total_mapped}"
)
print("Block tracking count check: PASS (575,061 blocks confirmed)")

# Calculate lifespan width (last_line_index - first_line_index)
lifespan_widths = np.fromiter(
    (block_last_line[b] - block_first_line[b] for b in block_first_line),
    dtype=np.int32,
    count=total_mapped
)

print("\n--- Block Lifespan Width Distribution (in lines) ---")
print(f"Min lifespan width    : {lifespan_widths.min()} lines")
print(f"Max lifespan width    : {lifespan_widths.max():,} lines")
print(f"Median lifespan width : {np.median(lifespan_widths):.1f} lines")
print(f"Mean lifespan width   : {lifespan_widths.mean():.2f} lines")
print(f"95th percentile       : {np.percentile(lifespan_widths, 95):.1f} lines")
print(f"99th percentile       : {np.percentile(lifespan_widths, 99):.1f} lines")
print(f"Single-line blocks    : {(lifespan_widths == 0).sum():,} ({(lifespan_widths == 0).mean() * 100:.2f}%)")

Streaming HDFS.log to map individual block lifespan boundaries...


Mapping lifespans: 100%|██████████| 11175629/11175629 [00:10<00:00, 1017001.03lines/s]


Total unique blocks mapped : 575,061
Block tracking count check: PASS (575,061 blocks confirmed)

--- Block Lifespan Width Distribution (in lines) ---
Min lifespan width    : 3 lines
Max lifespan width    : 6,971,706 lines
Median lifespan width : 1012220.0 lines
Mean lifespan width   : 1059813.05 lines
95th percentile       : 2661186.0 lines
99th percentile       : 2961378.6 lines
Single-line blocks    : 0 (0.00%)


**Findings 1.3:**

Block lifespan mapping was completed for all 575,061 unique block identifiers, confirmed against the count established in `00_calibration.ipynb`. The observed lifespan width distribution invalidates the feasibility of the single-cutoff, block-safe search strategy originally planned for Section 1.4: median lifespan width is 1,012,220 lines, approximately 9.1% of the full file, and maximum lifespan width is 6,971,706 lines, approximately 62.4% of the full file. No block has a zero-width (single-line) lifespan. Given that a block whose lifespan spans a large fraction of the file is, by construction, active across almost any interior line index, a genuinely block-safe single cutoff point satisfying zero straddling blocks does not exist in the interior of this file; such points exist only at the extreme start and end of the capture, where no practical partition can be placed. This finding required the split strategy in Sections 1.4 and 1.5 to be revised from a single-point block-safe cutoff search to a block-count-based chronological partition with explicit lifespan-leakage measurement and downstream censoring, as documented below.

### 1.4 Chronological Block-Count Partition (Revised Strategy)

The single-point block-safe cutoff search originally planned for this section was determined to be infeasible in Section 1.3: with a median block lifespan covering approximately 9.1% of the file, no interior line index exists at which zero blocks are straddling. The strategy is revised to the following, decided before the partition is computed in order to avoid selecting a boundary based on how the resulting split looks: every block is ordered chronologically by the line index of its first appearance, and this ordered list is partitioned directly by block count at the fixed 70/15/15 ratio. This guarantees the block-count ratio is met exactly, by construction, rather than approximately.

This section establishes block group membership only. It does not, by itself, establish that a block's underlying log lines are confined to its own split's line range; a block assigned to train by this rule may still have later-occurring lines that fall physically within the val or test line range, given the lifespan widths found in Section 1.3. This is measured explicitly and resolved in Section 1.5, not silently assumed away here.

In [4]:
from pathlib import Path
import numpy as np

# Sort all unique blocks chronologically by their earliest appearance (first_line_index)
sorted_blocks_by_start = sorted(
    block_first_line.items(),
    key=lambda x: x[1]
)

total_blocks = len(sorted_blocks_by_start)
print(f"Total unique blocks to partition: {total_blocks:,}")

# Target block partition counts based on the fixed 70/15/15 ratio
target_train_blocks = int(round(0.70 * total_blocks))
target_val_blocks = int(round(0.15 * total_blocks))
target_test_blocks = total_blocks - target_train_blocks - target_val_blocks

idx_train_val_split = target_train_blocks
idx_val_test_split = target_train_blocks + target_val_blocks

train_block_tuples = sorted_blocks_by_start[:idx_train_val_split]
val_block_tuples = sorted_blocks_by_start[idx_train_val_split:idx_val_test_split]
test_block_tuples = sorted_blocks_by_start[idx_val_test_split:]

train_block_set = {b[0] for b in train_block_tuples}
val_block_set = {b[0] for b in val_block_tuples}
test_block_set = {b[0] for b in test_block_tuples}

# Boundary line anchors (earliest line index where each split begins acquiring blocks)
cutoff_train_first_line = train_block_tuples[0][1]
cutoff_val_first_line = val_block_tuples[0][1]
cutoff_test_first_line = test_block_tuples[0][1]

print("\n--- Chronological Block-Arrival Split Boundaries ---")
print(f"Train Blocks: {len(train_block_set):,} ({(len(train_block_set)/total_blocks)*100:.2f}%) | Earliest start: Line {cutoff_train_first_line:,}")
print(f"Val Blocks  : {len(val_block_set):,} ({(len(val_block_set)/total_blocks)*100:.2f}%) | Earliest start: Line {cutoff_val_first_line:,}")
print(f"Test Blocks : {len(test_block_set):,} ({(len(test_block_set)/total_blocks)*100:.2f}%) | Earliest start: Line {cutoff_test_first_line:,}")


Total unique blocks to partition: 575,061

--- Chronological Block-Arrival Split Boundaries ---
Train Blocks: 402,543 (70.00%) | Earliest start: Line 1
Val Blocks  : 86,259 (15.00%) | Earliest start: Line 8,041,567
Test Blocks : 86,259 (15.00%) | Earliest start: Line 9,609,370


**Findings 1.4:**

Sorting all 575,061 blocks by first-appearance line index and partitioning directly by block count at the 70/15/15 ratio yields, by construction, an exact block-count split: 402,543 train blocks (70.00%), 86,259 validation blocks (15.00%), and 86,259 test blocks (15.00%). The validation split begins at line 8,041,567 and the test split begins at line 9,609,370. Unlike the abandoned single-cutoff approach, no shift-from-target or sanity-guard threshold applies here, since the partition is defined directly on block order rather than approximated toward a line-index target. This section confirms only block group membership; it does not confirm that the underlying log lines of each block remain within their assigned split's line range, which Section 1.3's lifespan findings suggest is unlikely to hold universally and which is measured directly in Section 1.5.

### 1.5 Lifespan Leakage Audit and Censoring Boundary Derivation

This section measures, directly and explicitly, the extent to which the block-count partition from Section 1.4 produces blocks whose lifespan crosses a split boundary, rather than assuming the partition is safe by construction (it is not, per Section 1.3). For each boundary, the number of blocks in the earlier split whose last line extends past the later split's starting line is counted, along with the furthest such extension. This quantity is the direct, honest cost of using a block-count partition on a dataset where a block-safe single cutoff does not exist, and is reported explicitly rather than absorbed silently into an adjacent split.

Because this leakage cannot be eliminated by boundary placement alone, it is resolved downstream through censoring: each block's split assignment (from Section 1.4) is retained for labeling and grouping purposes, but the log lines usable for that block's feature construction in Phase 4 are restricted to the line-index range of its own assigned split. This section derives and persists those split line-range boundaries as an interim artifact, separate from the block-to-split assignment artifact, so that Phase 4 can apply the censoring rule without recomputing any of the audits performed in this notebook.

In [5]:
import json
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
INTERIM_DIR.mkdir(parents=True, exist_ok=True)
ASSIGNMENTS_PATH = INTERIM_DIR / "block_split_assignments.csv"
BOUNDARIES_PATH = INTERIM_DIR / "split_line_boundaries.json"

EXPECTED_BLOCKS = 575_061
EXPECTED_LINES = 11_175_629

# 1. Assign each block to a split based on chronological order of first appearance
# (the block-count partition established in Section 1.4)
block_split_assignment = {}
for b in train_block_set:
    block_split_assignment[b] = "train"
for b in val_block_set:
    block_split_assignment[b] = "val"
for b in test_block_set:
    block_split_assignment[b] = "test"

assert len(block_split_assignment) == EXPECTED_BLOCKS, (
    f"Assigned block count ({len(block_split_assignment)}) does not match expected total ({EXPECTED_BLOCKS})!"
)
print(f"Structural partition check: PASS ({EXPECTED_BLOCKS:,} blocks assigned; "
      f"disjointness holds by construction, this is not a leakage check).")

# 2. Measure cross-boundary lifespan leakage directly, rather than assuming safety
val_start = min(block_first_line[b] for b in val_block_set)
test_start = min(block_first_line[b] for b in test_block_set)

max_train_last = max(block_last_line[b] for b in train_block_set)
affected_train_blocks = sum(1 for b in train_block_set if block_last_line[b] > val_start)

max_val_last = max(block_last_line[b] for b in val_block_set)
affected_val_blocks = sum(1 for b in val_block_set if block_last_line[b] > test_start)

print("\n--- Chronological Lifespan Leakage Audit ---")
print(f"Train/Val boundary : val starts at line {val_start:,}; "
      f"{affected_train_blocks:,} / {len(train_block_set):,} train blocks "
      f"({affected_train_blocks/len(train_block_set)*100:.2f}%) extend past this line "
      f"(max reach: line {max_train_last:,}).")
print(f"Val/Test boundary  : test starts at line {test_start:,}; "
      f"{affected_val_blocks:,} / {len(val_block_set):,} val blocks "
      f"({affected_val_blocks/len(val_block_set)*100:.2f}%) extend past this line "
      f"(max reach: line {max_val_last:,}).")
print("\n[POLICY] Block split assignment (from Section 1.4) is retained as-is for labeling "
      "and grouping. Event-level leakage across boundaries is resolved downstream in Phase 4 "
      "via line-range censoring: each block's usable event sequence is truncated to lines "
      "falling within its own split's line-range boundary, derived below and persisted for "
      "Phase 4 to consume directly.")

# 3. Derive line-range boundaries for downstream censoring
train_end_line = val_start - 1
val_end_line = test_start - 1
test_end_line = EXPECTED_LINES

split_line_boundaries = {
    "train": {"start_line": 1, "end_line": train_end_line},
    "val": {"start_line": train_end_line + 1, "end_line": val_end_line},
    "test": {"start_line": val_end_line + 1, "end_line": test_end_line},
    "leakage_audit": {
        "train_val_affected_blocks": affected_train_blocks,
        "train_val_max_train_reach_line": max_train_last,
        "val_test_affected_blocks": affected_val_blocks,
        "val_test_max_val_reach_line": max_val_last,
        "censoring_policy": (
            "Phase 4 must truncate each block's event sequence to lines within its "
            "assigned split's [start_line, end_line] range; events beyond this range "
            "must be excluded, not reassigned to the neighboring split."
        ),
    },
}

# 4. Realized split ratios based on the contiguous, censored line-range boundaries
print("\n--- Realized Split Ratios (Block Count vs. Censored Line Range) ---")
print(f"{'Split':<8} | {'Block Count':<12} | {'Block %':<8} | {'Line Range Width':<18} | {'Line %':<8}")
print("-" * 68)
for sp, b_set in [("train", train_block_set), ("val", val_block_set), ("test", test_block_set)]:
    line_width = split_line_boundaries[sp]["end_line"] - split_line_boundaries[sp]["start_line"] + 1
    print(f"{sp:<8} | {len(b_set):<12,} | {len(b_set)/EXPECTED_BLOCKS*100:<7.2f}% | "
          f"{line_width:<18,} | {line_width/EXPECTED_LINES*100:<7.2f}%")

# 5. Persist interim artifacts
df_assignments = pd.DataFrame(list(block_split_assignment.items()), columns=["block_id", "split"])
df_assignments.to_csv(ASSIGNMENTS_PATH, index=False)

with open(BOUNDARIES_PATH, "w", encoding="utf-8") as f:
    json.dump(split_line_boundaries, f, indent=2)

print(f"\nPersisted block assignment artifact : {ASSIGNMENTS_PATH.relative_to(PROJECT_ROOT)} ({len(df_assignments):,} blocks)")
print(f"Persisted line boundary artifact    : {BOUNDARIES_PATH.relative_to(PROJECT_ROOT)}")


Structural partition check: PASS (575,061 blocks assigned; disjointness holds by construction, this is not a leakage check).

--- Chronological Lifespan Leakage Audit ---
Train/Val boundary : val starts at line 8,041,567; 45,407 / 402,543 train blocks (11.28%) extend past this line (max reach: line 11,175,629).
Val/Test boundary  : test starts at line 9,609,370; 35,808 / 86,259 val blocks (41.51%) extend past this line (max reach: line 11,175,628).

[POLICY] Block split assignment (from Section 1.4) is retained as-is for labeling and grouping. Event-level leakage across boundaries is resolved downstream in Phase 4 via line-range censoring: each block's usable event sequence is truncated to lines falling within its own split's line-range boundary, derived below and persisted for Phase 4 to consume directly.

--- Realized Split Ratios (Block Count vs. Censored Line Range) ---
Split    | Block Count  | Block %  | Line Range Width   | Line %  
----------------------------------------------

**Findings 1.5:**

The lifespan leakage audit confirms that the block-count partition from Section 1.4 is not line-safe, consistent with the infeasibility finding of Section 1.3. At the train/validation boundary (validation starts at line 8,041,567), 45,407 of 402,543 train blocks (11.28%) have lines extending past this point, with the furthest-reaching train block extending to line 11,175,629, the final line of the file; this single block's lifespan spans the entire validation and test windows. At the validation/test boundary (test starts at line 9,609,370), 35,808 of 86,259 validation blocks (41.51%) have lines extending past this point, with the furthest-reaching validation block extending to line 11,175,628. The validation/test boundary is proportionally more affected than the train/validation boundary, consistent with validation being a narrower window (15% of blocks) in which a given block is more likely to still be active when the next boundary arrives.

Given the scale of this leakage, particularly the 41.51% figure at the validation/test boundary, no attempt was made to report it as an acceptable approximation; it is resolved through the censoring policy defined in the description above. The derived, contiguous censoring boundaries are: train lines 1 to 8,041,566 (8,041,566 lines, 71.96% of the file), validation lines 8,041,567 to 9,609,369 (1,567,803 lines, 14.03%), and test lines 9,609,370 to 11,175,629 (1,566,260 lines, 14.01%). These line-range percentages differ modestly from the exact 70/15/15 block-count ratio, which is expected: the censoring boundary is anchored to the earliest-starting block of each subsequent split, not to a line-count target, and blocks with wide lifespans starting early in train inflate its line-range share correspondingly.

Both `data/interim/block_split_assignments.csv` (575,061 rows, block-to-split membership) and `data/interim/split_line_boundaries.json` (per-split line ranges and the leakage audit figures above) were persisted. Phase 4 must read the boundaries artifact and truncate each block's event sequence to its own split's line range at feature-construction time; this notebook does not perform that truncation itself, since doing so here would require re-deriving Phase 4's feature representation prematurely.

### 1.6 Phase 1 Summary

The raw log's chronological integrity was confirmed in Section 1.1 (zero timestamp inversions across 132,373 distinct timestamps) and is sound enough to support treating line order as a valid ordering basis for splitting. However, the split boundary itself required a methodological revision mid-phase: the block lifespan mapping in Section 1.3 established that a single-point, fully block-safe cutoff does not exist in this dataset, since a substantial share of blocks remain active across large portions of the file (median lifespan approximately 9.1% of the file, maximum 62.4%). The strategy was revised from a single-cutoff search to a direct block-count partition at the exact 70/15/15 ratio (Section 1.4), combined with an explicit measurement of the resulting lifespan leakage at both boundaries rather than an assumption of safety (Section 1.5).

That measurement found leakage at both boundaries: 11.28% of train blocks extend past the start of the validation split, and 41.51% of validation blocks extend past the start of the test split, with the single most extreme case spanning the entire file. This leakage is not eliminated by boundary placement; it is instead resolved by a censoring policy handed off to Phase 4, under which each block's usable event sequence is truncated to the line-range boundary of its own assigned split. This decision, and the leakage figures that motivated it, are recorded here rather than absorbed silently, consistent with the project's honest-reporting principle: the exact 70/15/15 ratio applies to block count by construction, while the realized line-range ratio (71.97% / 14.03% / 14.02%) is a secondary, expected consequence of anchoring split boundaries to block arrival rather than to line count directly.

Two interim artifacts were persisted as the output of this phase: `data/interim/block_split_assignments.csv`, mapping each of the 575,061 blocks to its split, and `data/interim/split_line_boundaries.json`, defining each split's censored line range together with the leakage audit figures. On this basis, the raw data and its split structure are considered sound and sufficient to proceed to `02_log_parsing.ipynb`, with the explicit requirement that the Drain template dictionary be fit exclusively on lines falling within the train and validation line ranges defined here, and that the persisted censoring boundaries, not the raw block assignment alone, govern which lines are eligible for use in each split from Phase 2 onward.